# DP2 cluster overlap: sky map

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib


## Introduction

This notebook loads `dp2_cluster_overlap.csv`, keeps clusters that overlap the **DP2** survey region (`is_in_dp2`), and plots their positions on an all-sky map (Mollweide projection, ICRS RA/Dec in degrees).

- Load the cluster overlap CSV and resolve paths from the repo root or `_tutorials/`.
- Filter clusters overlapping the DP2 footprint (`is_in_dp2`).
- Plot all-sky Mollweide maps and compare redshift, richness, mass, and scaling-relation diagnostics.


## 1.0 Set Up


In [ ]:
### Import packages + setup common variables and useful functions
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make pretty
plt.rcParams.update({'font.size': 16})

%matplotlib inline


def resolve_data_csv() -> Path:
    """Find CSV whether the kernel cwd is the repo root or `_tutorials/`."""
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "_data" / "dp2_cluster_overlap.csv"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"dp2_cluster_overlap.csv not found under {Path.cwd()} or parent; "
        "open this notebook from `dp2-clusters-shear-mapping` or `_tutorials/`."
    )


csv_path = resolve_data_csv()
df = pd.read_csv(csv_path)
df.head()

## 2.0 Load catalog and DP2 subset

Read `dp2_cluster_overlap.csv` and keep rows with `is_in_dp2`.


In [ ]:
# Normalize boolean column (CSV may be parsed as bool or as strings)
in_dp2 = df["is_in_dp2"]
if in_dp2.dtype == object:
    in_dp2 = in_dp2.astype(str).str.lower().isin(("true", "1", "yes"))
dp2 = df.loc[in_dp2].copy()
print(f"Total rows: {len(df):,}")
print(f"In DP2 footprint (is_in_dp2): {len(dp2):,}")

## 3.0 All-sky cluster maps

Mollweide projection in ICRS; Milky Way latitude curves for context.


In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord

# Mollweide: center the map at RA = 180°, Dec = 0° (origin = lon 0, lat 0)
RA_CENTER_DEG = 0.0


def ra_dec_to_mollweide_lonlat(ra_deg_arr, dec_deg_arr):
    """Plot longitude (rad) with RA_CENTER_DEG at x=0; latitude = Dec in rad."""
    ra = np.asarray(ra_deg_arr, dtype=float)
    dec = np.asarray(dec_deg_arr, dtype=float)
    lon_deg = (ra - RA_CENTER_DEG + 180.0) % 360.0 - 180.0
    lon = np.radians(lon_deg)
    lat = np.radians(dec)
    return lon, lat


def galactic_b_curve_mollweide(b_deg: float, n: int = 4000):
    l_deg = np.linspace(0.0, 360.0, n, endpoint=False)
    gal = SkyCoord(
        l=l_deg * u.deg,
        b=np.full(n, b_deg, dtype=float) * u.deg,
        frame="galactic",
    )
    icrs = gal.transform_to("icrs")
    return ra_dec_to_mollweide_lonlat(icrs.ra.deg, icrs.dec.deg)


def split_curve_at_lon_jumps(lon, lat, max_jump=np.deg2rad(95)):
    """Break a closed sky curve into segments across the [-pi, pi] longitude seam."""
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    jumps = np.where(np.abs(np.diff(lon)) > max_jump)[0] + 1
    if jumps.size == 0:
        return [(lon, lat)]
    out = []
    start = 0
    for j in jumps:
        out.append((lon[start:j], lat[start:j]))
        start = j
    out.append((lon[start:], lat[start:]))
    return out


def plot_cluster_sky_mollweide(catalog_df, title: str) -> None:
    """All-sky Mollweide: richness → marker size, redshift → color; Milky Way b = 0°, ±10°."""
    ra_deg = catalog_df["RA"].to_numpy(dtype=float)
    dec_deg = catalog_df["DEC"].to_numpy(dtype=float)
    z = catalog_df["redshift"].to_numpy(dtype=float)
    rich = catalog_df["richness"].to_numpy(dtype=float)

    lon_rad, lat_rad = ra_dec_to_mollweide_lonlat(ra_deg, dec_deg)

    r_lo, r_hi = np.nanpercentile(rich, [5, 95])
    rich_clip = np.clip(rich, r_lo, r_hi)
    size = 6 + 34 * (rich_clip - r_lo) / (r_hi - r_lo + 1e-12)

    fig = plt.figure(figsize=(14, 7), dpi=120)
    ax = fig.add_subplot(111, projection="mollweide")
    fig.patch.set_facecolor("white")

    sc = ax.scatter(
        lon_rad,
        lat_rad,
        s=size,
        c=z,
        cmap="viridis",
        vmin=np.nanpercentile(z, 1),
        vmax=np.nanpercentile(z, 99),
        alpha=0.75,
        linewidths=0,
    )

    for b_deg in (-10.0, 0.0, 10.0):
        lon_b, lat_b = galactic_b_curve_mollweide(b_deg)
        for lo, la in split_curve_at_lon_jumps(lon_b, lat_b):
            ax.plot(
                lo,
                la,
                color="black",
                linestyle="--" if b_deg == 0.0 else "-",
                linewidth=2.0 if b_deg == 0.0 else 1.2,
                zorder=6,
                solid_capstyle="round",
            )

    xticks_deg = np.array([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150])
    ax.set_xticks(np.radians(xticks_deg))
    ra_at_xticks = (xticks_deg + RA_CENTER_DEG) % 360.0
    ax.set_xticklabels([f"{ra:.0f}°" for ra in ra_at_xticks])
    ax.set_xlabel("RA (ICRS, °)")
    yticks_deg = np.array([-60, -30, 0, 30, 60])
    ax.set_yticks(np.radians(yticks_deg))
    ax.set_yticklabels([f"{d}°" for d in yticks_deg])
    ax.set_ylabel("Dec (ICRS, °)")

    ax.grid(True, color="black", linestyle="-", alpha=0.25)

    cb = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.06)
    cb.set_label("Redshift")

    ax.set_title(title, pad=14)
    plt.tight_layout()
    plt.show()


plot_cluster_sky_mollweide(dp2, f"Clusters in DP2 region (N = {len(dp2):,})")
plot_cluster_sky_mollweide(df, f"Full catalog — all clusters (N = {len(df):,})")


## 4.0 Redshift distributions

Photometric redshift histograms: full catalog vs DP2 subset.


In [ ]:
z_all = df["redshift"].to_numpy(dtype=float)
z_dp2 = dp2["redshift"].to_numpy(dtype=float)
z_all = z_all[np.isfinite(z_all)]
z_dp2 = z_dp2[np.isfinite(z_dp2)]

z_lo = float(min(z_all.min(), z_dp2.min()))
z_hi = float(max(z_all.max(), z_dp2.max()))
bins = np.linspace(z_lo, z_hi, 80)

# Cohesive blue palette
blue_fill = "#A3C4E8"
blue_fill_edge = "#6B9BD1"
blue_line = "#1F4F8F"

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.hist(
    z_all,
    bins=bins,
    histtype="stepfilled",
    alpha=0.5,
    color=blue_fill,
    edgecolor=blue_fill_edge,
    linewidth=0.9,
    density=True,
    label=f"All clusters\n(N = {len(df):,})",
)

n_dp2, edges = np.histogram(z_dp2, bins=bins, density=True)
z_centers = 0.5 * (edges[:-1] + edges[1:])
ax.plot(
    z_centers,
    n_dp2,
    drawstyle="steps-mid",
    color=blue_line,
    linewidth=2.4,
    label=f"DP2\n(N = {len(dp2):,})",
)

ax.set_xlabel("Redshift")
ax.set_ylabel("Density")
ax.legend(frameon=True)
ax.set_title("Redshift distribution")
plt.tight_layout()
plt.show()


## 5.0 Richness distributions

Richness histograms with the same styling as redshift.


In [ ]:
r_all = df["richness"].to_numpy(dtype=float)
r_dp2 = dp2["richness"].to_numpy(dtype=float)
r_all = r_all[np.isfinite(r_all)]
r_dp2 = r_dp2[np.isfinite(r_dp2)]

r_lo = float(min(np.percentile(r_all, 0.5), np.percentile(r_dp2, 0.5)))
r_hi = float(max(np.percentile(r_all, 99.5), np.percentile(r_dp2, 99.5)))
bins = np.linspace(r_lo, r_hi, 80)

blue_fill = "#A3C4E8"
blue_fill_edge = "#6B9BD1"
blue_line = "#1F4F8F"

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.hist(
    r_all,
    bins=bins,
    histtype="stepfilled",
    alpha=0.5,
    color=blue_fill,
    edgecolor=blue_fill_edge,
    linewidth=0.9,
    density=True,
    label=f"All clusters (N = {len(df):,})",
)

n_dp2, edges = np.histogram(r_dp2, bins=bins, density=True)
r_centers = 0.5 * (edges[:-1] + edges[1:])
ax.plot(
    r_centers,
    n_dp2,
    drawstyle="steps-mid",
    color=blue_line,
    linewidth=2.4,
    label=f"DP2 (N = {len(dp2):,})",
)

ax.set_xlabel("Richness")
ax.set_ylabel("Density")
ax.set_yscale('log')
ax.legend(frameon=True)
ax.set_title("Richness distribution")
plt.tight_layout()
plt.show()


## 6.0 Mass and scaling relations

`M200m` histograms and scatter plots vs richness and redshift.


In [ ]:
# Plot histogram for cluster mass (M200m)

m_all = df["M200m"].to_numpy(dtype=float)
m_dp2 = dp2["M200m"].to_numpy(dtype=float)
m_all = m_all[np.isfinite(m_all) & (m_all > 0)]
m_dp2 = m_dp2[np.isfinite(m_dp2) & (m_dp2 > 0)]

m_lo = float(min(np.percentile(m_all, 0.5), np.percentile(m_dp2, 0.5)))
m_hi = float(max(np.percentile(m_all, 99.5), np.percentile(m_dp2, 99.5)))
mass_bins = np.linspace(m_lo, m_hi, 80)

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.hist(
    m_all,
    bins=mass_bins,
    histtype="stepfilled",
    alpha=0.5,
    color=blue_fill,
    edgecolor=blue_fill_edge,
    linewidth=0.9,
    density=True,
    label=f"All clusters (N = {len(m_all):,})",
)

n_dp2_m, mass_edges = np.histogram(m_dp2, bins=mass_bins, density=True)
mass_centers = 0.5 * (mass_edges[:-1] + mass_edges[1:])
ax.plot(
    mass_centers,
    n_dp2_m,
    drawstyle="steps-mid",
    color=blue_line,
    linewidth=2.4,
    label=f"DP2 (N = {len(m_dp2):,})",
)

ax.set_xlabel(r"$M_{200m}$ [M$_\odot$]")
ax.set_ylabel("Density")
ax.set_yscale('log')
ax.legend(frameon=True)
ax.set_title(r"$M_{200m}$ (cluster mass) distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Plot mass (M200m) vs richness relation

fig, ax = plt.subplots(figsize=(10, 6), dpi=120)

# All clusters
ax.scatter(
    df["richness"], 
    df["M200m"], 
    s=24, 
    color=blue_fill, 
    edgecolor=blue_fill_edge, 
    alpha=0.4, 
    label="All clusters"
)

# Highlight DP2 clusters, if available
if "richness" in dp2.columns and "M200m" in dp2.columns:
    ax.scatter(
        dp2["richness"],
        dp2["M200m"],
        s=40,
        color=blue_line,
        edgecolor="k",
        alpha=0.9,
        label="DP2 clusters"
    )

ax.set_xlabel(r"Richness $(\lambda)$")
ax.set_ylabel(r"$M_{200m}$ [M$_\odot$]")
ax.set_title(r"Cluster Mass ($M_{200m}$) vs Richness")
ax.set_xscale("linear")
ax.set_yscale("log")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:

# Plot redshift vs cluster mass (M200m)
fig, ax = plt.subplots(figsize=(10, 6), dpi=120)

# All clusters
ax.scatter(
    df["redshift"],
    df["M200m"],
    s=24,
    color=blue_fill,
    edgecolor=blue_fill_edge,
    alpha=0.4,
    label="All clusters"
)

# Highlight DP2 clusters, if available
if "redshift" in dp2.columns and "M200m" in dp2.columns:
    ax.scatter(
        dp2["redshift"],
        dp2["M200m"],
        s=40,
        color=blue_line,
        edgecolor="k",
        alpha=0.9,
        label="DP2 clusters"
    )

ax.set_xlim(0,2.0)
ax.set_xlabel(r"Redshift ($z$)")
ax.set_ylabel(r"$M_{200m}$ [M$_\odot$]")
ax.set_title(r"Redshift vs Cluster Mass ($M_{200m}$)")
ax.set_yscale("log")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

